# LIMINA -- 05. Penilaian Pasar Hari Ini dan Artefak Produk

Menilai SELURUH cakupan emiten dengan model yang tersimpan di `artifacts/`,
memakai data paling baru di `data/raw/` (hasil notebook 01), lalu menulis
`artifacts/scores.json` -- keluaran yang dikonsumsi dashboard/produk.

Beda dari notebook 04: notebook itu mengukur performa historis (backtest),
notebook ini menghasilkan peringkat risiko UNTUK HARI INI. `is_event_90d`
pada potret hari ini selalu 0 (placeholder) karena kita tidak bisa tahu apa
yang akan terjadi 90 hari ke depan -- ini BUKAN evaluasi, jangan dipakai
menghitung Precision@20 dari keluaran notebook ini.

In [ ]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json
from datetime import date

import pandas as pd

from limina import artifacts_io, baselines, config, model_registry, models, raw_ingest

df_qf = pd.read_csv(config.RAW_DIR / f"{config.TABEL_QUARTERLY_FINANCIALS}.csv")
df_dt = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_TRANSACTION}.csv")
df_dfu = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_FULL_UNIVERSE_CLOSE}.csv")
df_ff = pd.read_csv(config.RAW_DIR / f"{config.TABEL_FREE_FLOAT_SNAPSHOT}.csv")
df_co = pd.read_csv(config.RAW_DIR / f"{config.TABEL_COMPANY_OVERVIEW}.csv")

df_harga = raw_ingest.gabungkan_harga(df_dt, df_dfu)
peta_sektor = raw_ingest.bangun_peta_sektor(df_ff, df_co)
peta_board = raw_ingest.bangun_peta_board(df_co)
peta_nama = dict(zip(df_co["symbol"], df_co["company_name"])) if "company_name" in df_co.columns else {}

symbols_universe = sorted(set(df_qf["symbol"]) | set(df_co["symbol"]) | set(df_harga["symbol"]))
hari_ini = date.today().strftime("%Y-%m-%d")
print(f"Menilai {len(symbols_universe)} emiten pada {hari_ini}")

## 1. Bangun potret pasar hari ini

In [ ]:
potret_live = raw_ingest.bangun_snapshot_pasar(
    hari_ini, symbols_universe, df_qf, df_harga, peta_sektor, peta_board=peta_board,
)
potret_live["nama"] = potret_live["symbol"].map(peta_nama).fillna("")
potret_live["data_terakhir"] = potret_live["feature_max_source_date"].astype(str)

print(f"{len(potret_live)} baris, {int((potret_live['data_complete'] == 0).sum())} tidak lengkap")

## 2. Bandingkan dengan riwayat 30 hari lalu (arah_30h, delta_30h)

Kalau notebook 03 belum pernah berhasil menyimpan model (mis. data belum cukup untuk dilatih -- lihat diagnosa notebook 02), cell berikut otomatis memakai Kandidat 4 (rule-based, `limina/baselines.py`) sebagai skor sementara. Kandidat ini TIDAK memerlukan pelatihan sama sekali, jadi tetap memberi angka risiko untuk emiten yang datanya lengkap hari ini, sambil menunggu cakupan data cukup untuk regresi logistik.

`artifacts/riwayat_skor.csv` terkumpul otomatis setiap kali notebook ini jalan (lihat cell terakhir). Sebelum ada sekitar 30 hari riwayat terkumpul, seluruh emiten akan tertulis "stabil" -- ini bukan bug, hanya belum ada pembanding untuk dihitung.

In [ ]:
model_tersedia = model_registry.model_tersedia()

if model_tersedia:
    model_lr, scaler, median_latih = model_registry.muat_model_terlatih()
    skor_mentah = models.skor_kandidat_1(model_lr, scaler, potret_live, median_latih)
else:
    print(
        "Model regresi logistik belum tersedia (notebook 03 belum pernah selesai "
        "melatih -- lihat diagnosa cakupan data di notebook 02, atau "
        "docs/rancangan/AMBA-struktur-model-dan-algoritma.md bagian 6 soal "
        "rule-based sebagai jalur cadangan). Memakai Kandidat 4 (rule-based, TIDAK "
        "memerlukan pelatihan) sebagai skor SEMENTARA. Begitu notebook 03 berhasil "
        "menyimpan model, jalankan ulang notebook ini untuk beralih otomatis."
    )
    skor_mentah = potret_live.apply(baselines.skor_rule_based_mentah, axis=1)

persentil_hari_ini = skor_mentah.rank(pct=True) * 100

if config.PATH_RIWAYAT_SKOR.exists():
    riwayat_skor = pd.read_csv(config.PATH_RIWAYAT_SKOR, parse_dates=["tanggal"])
else:
    riwayat_skor = pd.DataFrame(columns=["symbol", "tanggal", "persentil"])

TOLERANSI_HARI = 5  # +-5 hari di sekitar 30 hari lalu, supaya tetap ketemu walau siklus tidak jalan persis tiap hari
AMBANG_PERUBAHAN = 5  # poin persentil, di bawah ini dianggap "stabil"
target_30h = pd.Timestamp(hari_ini) - pd.Timedelta(days=30)

arah_30h, delta_30h = [], []
for symbol, persentil_now in zip(potret_live["symbol"], persentil_hari_ini):
    riwayat_symbol = riwayat_skor[riwayat_skor["symbol"] == symbol].copy()
    if len(riwayat_symbol) == 0:
        arah_30h.append("stabil")
        delta_30h.append(0.0)
        continue
    riwayat_symbol["jarak"] = (riwayat_symbol["tanggal"] - target_30h).abs()
    terdekat = riwayat_symbol.sort_values("jarak").iloc[0]
    if terdekat["jarak"] > pd.Timedelta(days=TOLERANSI_HARI):
        arah_30h.append("stabil")
        delta_30h.append(0.0)
        continue
    delta = float(persentil_now - terdekat["persentil"])
    delta_30h.append(delta)
    arah_30h.append("naik" if delta > AMBANG_PERUBAHAN else "turun" if delta < -AMBANG_PERUBAHAN else "stabil")

potret_live["arah_30h"] = arah_30h
potret_live["delta_30h"] = delta_30h
print(pd.Series(arah_30h).value_counts())

## 3. Tulis scores.json

Kalau memakai jalur rule-based sementara (bagian 2), `indikator_dominan` dan kontribusi per-indikator dikosongkan -- itu hasil dekomposisi koefisien regresi logistik (`limina/models.py::kontribusi_indikator`), tidak berlaku untuk skor rule-based. `persentil`/`kategori` tetap valid untuk kedua jalur.

In [ ]:
with open(config.PATH_JENDELA) as f:
    jendela = json.load(f)

jumlah_positif_latih = int(pd.read_csv(config.PATH_PANEL)["is_event_90d"].sum()) if config.PATH_PANEL.exists() else 0
cutoff_latih_str = jendela["cutoff_latih"]
output_path = config.ARTIFACTS_DIR / "scores.json"

if model_tersedia:
    data_scores = model_registry.hasilkan_scores_json(
        potret_live,
        jumlah_sampel_positif_latih=jumlah_positif_latih,
        dilatih_pada=f"peristiwa sebelum {cutoff_latih_str}",
        k_top=config.K_TOP,
    )
else:
    df_skor = potret_live.copy()
    df_skor["skor"] = skor_mentah
    df_skor["persentil"] = persentil_hari_ini
    k_efektif = min(config.K_TOP, len(df_skor)) if len(df_skor) else 0
    ambang_persentil = 100 * (1 - k_efektif / len(df_skor)) if len(df_skor) else 0.0
    ambang_nilai = float(skor_mentah.quantile(ambang_persentil / 100)) if len(df_skor) else 0.0
    data_scores = artifacts_io.bangun_scores_json(
        df_skor,
        {},  # rule-based: tidak ada dekomposisi kontribusi per-indikator seperti regresi logistik
        jenis_model="rule_based",
        dilatih_pada="tidak dilatih -- skor rule-based memakai bobot tetap, lihat limina/baselines.py",
        jumlah_sampel_positif_latih=jumlah_positif_latih,
        ambang_persentil=float(ambang_persentil),
        ambang_nilai=ambang_nilai,
    )
    artifacts_io.tulis_json(data_scores, output_path)

print(f"scores.json ditulis: {output_path}")
print(f"Cakupan: {data_scores['cakupan']}")
print("\n20 emiten berisiko tertinggi:")
display(
    pd.DataFrame(data_scores["emiten"])
    .sort_values("persentil", ascending=False)
    .head(20)[["symbol", "nama", "sektor", "papan", "status", "persentil", "kategori", "indikator_dominan"]]
)

## 4. Tambahkan hari ini ke riwayat_skor.csv (untuk perbandingan 30 hari berikutnya)

In [ ]:
baris_baru = pd.DataFrame({
    "symbol": potret_live["symbol"],
    "tanggal": pd.Timestamp(hari_ini),
    "persentil": persentil_hari_ini.values,
})
riwayat_skor_baru = pd.concat([riwayat_skor, baris_baru], ignore_index=True)
riwayat_skor_baru = riwayat_skor_baru.drop_duplicates(subset=["symbol", "tanggal"], keep="last")

config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
riwayat_skor_baru.to_csv(config.PATH_RIWAYAT_SKOR, index=False)
print(f"riwayat_skor.csv: {len(riwayat_skor_baru)} baris total setelah menambahkan hari ini")
print("\nSiklus selesai. Jalankan ulang notebook 01-05 secara berkala (harian/mingguan) untuk memperbarui model dan skor.")